# Clínica - Projeto Itaú

In [1]:
import json
import os
import re
import unicodedata
import random
import pandas as pd
import asyncio
import hashlib
import time
from pathlib import Path

import nest_asyncio

Entrada e normalização
Leitura do CSV e ajuste inicial de texto/codificação.

In [ ]:
# LINHA DE SELEÇÃO DO INPUT
df = pd.read_csv("dataset_clinica_20261.csv", encoding="utf-8") # 4s para carregar

def corrigir_mojibake(valor):
    if isinstance(valor, str) and ("Ã" in valor or "Â" in valor):
        try:
            return valor.encode("latin1").decode("utf-8")
        except UnicodeError:
            return valor
    return valor

colunas_texto = df.select_dtypes(include="object").columns
df[colunas_texto] = df[colunas_texto].apply(lambda col: col.map(corrigir_mojibake))

if "magistrado" in df.columns:
    df["magistrado"] = df["magistrado"].astype(str).str.strip().str.upper()

display(df.head())
df.info()

C:\Users\gabriel\AppData\Local\Temp\ipykernel_20004\2772562578.py:12: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  colunas_texto = df.select_dtypes(include="object").columns


Preparação da amostra para análise com IA

100 casos aleatórios (reproduzíveis via `random_state=42`).

In [ ]:
df_curto = df.copy()
df_curto = df_curto.sample(n=1000, random_state=42).reset_index(drop=True)

In [ ]:
df_curto.shape

Variáveis para extração

- justiça_gratuita (sim/não)
- rito_processual (Juizado Especial / Procedimento Comum)
- tipo_acao (fraude/golpe, cobrança indevida, empréstimo não reconhecido, revisão contratual)
- contato_previo_banco (sim/não)
- canal_contato (SAC, Ouvidoria, Procon, Reclame Aqui, Agência, não identificado)
- mencao_reclame_aqui (sim/não)
- boletim_de_ocorrencia (sim/não)
- resultado_julgamento (procedente, improcedente, parcialmente procedente, extinto)
- culpa_atribuida (banco, consumidor, terceiro, compartilhada)
- valor_danos_morais (float, R$)
- valor_danos_materiais (float, R$)


## Extração com OpenAI — async
Extrai as variáveis em JSON com cache e retry.

In [ ]:
nest_asyncio.apply()

try:
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm

from openai import AsyncOpenAI
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("Defina OPENAI_API_KEY antes de executar esta célula.")

async_client = AsyncOpenAI(api_key=api_key)
modelo_openai = "gpt-4.1-mini"
MAX_CONCORRENTE = 20
SALVAR_CACHE_A_CADA = 50

# --- Cache local ---
CACHE_PATH = Path("cache") / "cache_openai.json"
CACHE_PATH.parent.mkdir(exist_ok=True)

In [ ]:
def _carregar_cache():
    if CACHE_PATH.exists():
        with open(CACHE_PATH, "r", encoding="utf-8") as f:
            return json.load(f)
    return {}

def _salvar_cache(cache):
    with open(CACHE_PATH, "w", encoding="utf-8") as f:
        json.dump(cache, f, ensure_ascii=False, indent=2)

cache_local = _carregar_cache()
print(f"Cache carregado: {len(cache_local)} entradas existentes")

# --- Prompts ---
prompt_sistema = (
    "Você é um analista jurídico especializado em processos cíveis contra bancos. "
    "Extraia informações estruturadas de decisões judiciais em português. "
    "Responda APENAS com JSON válido, sem texto adicional."
)

template_prompt = """Analise o documento judicial abaixo e extraia as variáveis indicadas.

PASSO 1 — TRIAGEM:
Identifique o tipo do documento:
- "sentença": decisão que resolve o processo (total ou parcialmente)
- "despacho": ato de impulso processual sem conteúdo decisório
- "decisão interlocutória": decisão que não encerra o processo
- "outro": embargos de declaração, cumprimento de sentença, etc.

Se tipo_documento != "sentença", retorne APENAS:
{{"tipo_documento": "<valor>", "fora_do_escopo": true}}

PASSO 2 — EXTRAÇÃO (apenas para sentenças):

VARIÁVEIS:
1. tipo_documento: "sentença"
2. fora_do_escopo: false
3. justica_gratuita: ("sim" / "não" / "não identificado")
   Retorne "sim" se o texto mencionar: "Justiça Gratuita", "gratuidade da justiça",
   "gratuidade processual", "benefícios da AJG", "benefícios da assistência judiciária",
   ou se o cabeçalho listar "Justiça Gratuita" como campo do processo.
   Retorne "não" apenas se o texto mencionar explicitamente que a gratuidade foi
   indeferida ou que a parte recolheu custas.
   Retorne "não identificado" se não houver qualquer menção ao tema.
4. rito_processual: ("Juizado Especial" / "Procedimento Comum")
5. tipo_acao:
   - "empréstimo não reconhecido": autor nega ter contratado empréstimo consignado
   - "fraude em conta ou cartão": golpe, transação não autorizada em conta/cartão (motoboy, falsa central, boa noite Cinderela, etc.)
   - "revisão contratual": autor questiona taxas ou cláusulas de contrato que reconhece ter firmado
   - "cobrança indevida": cobrança de tarifa, serviço ou parcela não contratada
   - "superendividamento": pedido de repactuação com base na Lei 14.181/2021
   - "fora_do_escopo": cumprimento de sentença, extinção por inércia, acordo homologado, litispendência
   - "outro": não se encaixa em nenhuma categoria acima
6. contato_previo_banco: ("sim" / "não")
   IMPORTANTE: Este campo aceita APENAS "sim" ou "não". Nunca retorne "não identificado".
   Retorne "sim" APENAS se a petição inicial menciona que o autor contatou
   o banco ANTES de decidir processar, com objetivo de reclamar ou resolver o problema
   (ex: ligou para SAC, foi à agência, registrou no Procon).
   NÃO contar: contato feito para registrar a fraude já ocorrida, para tentar
   estornar valor após o golpe, ou qualquer contato posterior ao evento danoso.
   Em todos os outros casos (incluindo quando não há menção), retorne "não".
7. canal_contato: ("SAC" / "Ouvidoria" / "Procon" / "Reclame Aqui" / "Agência" / "não identificado")
   Retorne apenas se mencionado explicitamente no texto como canal de contato prévio.
8. mencao_reclame_aqui: ("sim" / "não")
9. boletim_de_ocorrencia: ("sim" / "não")
10. resultado_julgamento:
    - "procedente"
    - "parcialmente procedente"
    - "improcedente"
    - "extinto sem mérito": extinção por inércia, desistência, falta de pressuposto, litispendência
    - "extinto com mérito prescrição": extinção por prescrição ou decadência (art. 487 II)
    - "extinto acordo": homologação de transação entre as partes
11. culpa_atribuida: ("banco" / "consumidor" / "terceiro" / "compartilhada" / "não identificado")
    REGRA:
    - improcedente → "consumidor" (salvo exceção explícita no texto)
    - procedente ou parcialmente procedente → "banco" (salvo exceção explícita)
    - extinto (qualquer tipo) → "não identificado"
12. valor_danos_morais: número em reais
    0.0 = não condenado | -1.0 = condenado mas valor a apurar em liquidação
13. valor_danos_materiais: número em reais
    0.0 = não condenado | -1.0 = condenado mas valor a apurar em liquidação
14. repetição_indébito: ("simples" / "dobro" / "não aplicável" / "não identificado")
    Refere-se à devolução de valores cobrados indevidamente (art. 42 CDC).

REGRAS GERAIS:
- Retorne "não" ou "não identificado" quando não houver menção explícita no texto.
- Nunca infira o que não está escrito.
- Quando a sentença julgar múltiplos contratos com resultados diferentes,
  classifique pelo resultado majoritário ou use "parcialmente procedente".

Retorne exatamente este JSON:
{{"tipo_documento":"sentença","fora_do_escopo":false,"justica_gratuita":"...","rito_processual":"...","tipo_acao":"...","contato_previo_banco":"...","canal_contato":"...","mencao_reclame_aqui":"...","boletim_de_ocorrencia":"...","resultado_julgamento":"...","culpa_atribuida":"...","valor_danos_morais":0.0,"valor_danos_materiais":0.0,"repetição_indébito":"..."}}

Documento:
{decisao}"""

_campos_padrao = {
    "tipo_documento": "não identificado",
    "fora_do_escopo": False,
    "justica_gratuita": "não",
    "rito_processual": "não identificado",
    "tipo_acao": "não identificado",
    "contato_previo_banco": "não",
    "canal_contato": "não identificado",
    "mencao_reclame_aqui": "não",
    "boletim_de_ocorrencia": "não",
    "resultado_julgamento": "não identificado",
    "culpa_atribuida": "não identificado",
    "valor_danos_morais": 0.0,
    "valor_danos_materiais": 0.0,
    "repetição_indébito": "não identificado",
}

def _hash_decisao(texto):
    chave = texto[:500].strip()
    return hashlib.md5(chave.encode("utf-8")).hexdigest()

def _parse_json_seguro(conteudo):
    try:
        return json.loads(conteudo)
    except json.JSONDecodeError:
        m = re.search(r"\{.*\}", conteudo, flags=re.DOTALL)
        if m:
            return json.loads(m.group(0))
        raise

async def _chamar_decisao_async(idx, texto, semaforo, max_tentativas=4):
    prompt = template_prompt.format(decisao=texto)
    async with semaforo:
        for tentativa in range(max_tentativas):
            try:
                params = dict(
                    model=modelo_openai,
                    messages=[
                        {"role": "system", "content": prompt_sistema},
                        {"role": "user", "content": prompt},
                    ],
                    response_format={"type": "json_object"},
                )
                if not modelo_openai.startswith("o"):
                    params["temperature"] = 0

                resposta = await async_client.chat.completions.create(**params)
                dados = _parse_json_seguro(resposta.choices[0].message.content.strip())

                for campo in ("valor_danos_morais", "valor_danos_materiais"):
                    try:
                        dados[campo] = float(dados.get(campo) or 0.0)
                    except (ValueError, TypeError):
                        dados[campo] = 0.0

                return idx, {**_campos_padrao, **dados}

            except Exception as e:
                is_429 = "429" in str(e)
                is_last = tentativa == max_tentativas - 1
                if is_429 and not is_last:
                    espera = min(2 ** tentativa * 5, 60)
                    jitter = random.uniform(0, espera * 0.3)
                    await asyncio.sleep(espera + jitter)
                    continue
                return idx, {**_campos_padrao, "resultado_julgamento": f"ERRO: {str(e)}"}


async def processar_async(indices_api, decisoes, resultados_final):
    semaforo = asyncio.Semaphore(MAX_CONCORRENTE)
    tasks = [_chamar_decisao_async(i, decisoes[i], semaforo) for i in indices_api]
    pendentes_flush = 0

    with tqdm(total=len(tasks), desc="Extraindo (async)", unit="dec") as barra:
        for coro in asyncio.as_completed(tasks):
            idx, resultado = await coro
            resultados_final[idx] = resultado

            if not str(resultado.get("resultado_julgamento", "")).startswith("ERRO"):
                cache_local[_hash_decisao(decisoes[idx])] = resultado
                pendentes_flush += 1

            if pendentes_flush >= SALVAR_CACHE_A_CADA:
                _salvar_cache(cache_local)
                pendentes_flush = 0

            barra.update(1)

    if pendentes_flush > 0:
        _salvar_cache(cache_local)


# --- Separa cached dos que precisam de API (reprocessa entradas com ERRO) ---
decisoes = df_curto["decisao"].fillna("").tolist()
resultados_final = [None] * len(decisoes)

indices_api = []
for i, texto in enumerate(decisoes):
    hash_key = _hash_decisao(texto)
    resultado_em_cache = cache_local.get(hash_key)

    if resultado_em_cache is None:
        indices_api.append(i)
    elif str(resultado_em_cache.get("resultado_julgamento", "")).startswith("ERRO"):
        # Remove do cache para forçar nova chamada na próxima execução
        indices_api.append(i)
        del cache_local[hash_key]
    else:
        resultados_final[i] = resultado_em_cache

hits_cache = len(decisoes) - len(indices_api)
print(f"Cache hits: {hits_cache}/{len(decisoes)} | A processar: {len(indices_api)} | Concorrência: {MAX_CONCORRENTE}")

# --- Dispara tudo assincronamente ---
t_inicio = time.time()
await processar_async(indices_api, decisoes, resultados_final)
tempo_total = time.time() - t_inicio

print(f"\nConcluído em {tempo_total:.1f}s | Cache: {len(cache_local)}/{len(decisoes)}")
print(f"Referência síncrona: 7min 22s (442s) → speedup: {442/tempo_total:.1f}×")

# --- Monta df_curto: colunas originais + campos da IA com sufixo _ia ---
colunas_originais = [c for c in df_curto.columns if not c.endswith("_ia")]
df_curto = df_curto[colunas_originais]

df_ai = pd.DataFrame(resultados_final).rename(
    columns={c: f"{c}_ia" for c in _campos_padrao.keys()}
)

df_curto = pd.concat([df_curto.reset_index(drop=True), df_ai.reset_index(drop=True)], axis=1)

print(f"\nColunas finais ({len(df_curto.columns)}): {list(df_curto.columns)}")
df_curto.head(5)

Reprocessamento dos erros

In [ ]:
## Reprocessamento

# Identifica linhas com ERRO no resultado
mask_erros = df_curto['resultado_julgamento_ia'].astype(str).str.startswith('ERRO', na=False)
indices_erro = df_curto[mask_erros].index.tolist()
print(f"Linhas com ERRO para reprocessar: {len(indices_erro)}")

if indices_erro:
    # Pega os textos e limpa o cache dessas entradas (força nova chamada)
    decisoes_erro = df_curto.loc[indices_erro, 'decisao'].fillna('').tolist()
    for texto in decisoes_erro:
        hash_key = _hash_decisao(texto)
        if hash_key in cache_local:
            del cache_local[hash_key]

    # Reprocessa só os erros
    resultados_reprocess = [None] * len(decisoes_erro)
    indices_todos = list(range(len(decisoes_erro)))

    t_inicio = time.time()
    await processar_async(indices_todos, decisoes_erro, resultados_reprocess)
    tempo_total = time.time() - t_inicio
    print(f"Reprocessamento concluído em {tempo_total:.1f}s")

    # Atualiza df_curto com os novos resultados
    df_novos = pd.DataFrame(resultados_reprocess).rename(
        columns={c: f"{c}_ia" for c in _campos_padrao.keys()}
    )

    colunas_ia = [f"{c}_ia" for c in _campos_padrao.keys()]
    df_curto.loc[indices_erro, colunas_ia] = df_novos[colunas_ia].values

    # Verifica se ainda restam erros
    erros_restantes = df_curto['resultado_julgamento_ia'].astype(str).str.startswith('ERRO', na=False).sum()
    print(f"Erros restantes após reprocessamento: {erros_restantes}")
    pasta_saida = "output"
    os.makedirs(pasta_saida, exist_ok=True)
    # Salva o arquivo atualizado
    arquivo_saida = os.path.join(pasta_saida, "curto_ia.xlsx")
    df_curto.to_excel(arquivo_saida, index=False, sheet_name="Processos")
    print("Arquivo atualizado salvo: output/curto_ia.xlsx")
else:
    print("Nenhum erro encontrado — nada a reprocessar.")

## Classificação por Regex — baseline de comparação

Para cada variável extraída pela IA, criamos um classificador por regex aplicado ao texto bruto da decisão.
Objetivo: medir o **acordo IA vs regex**. Alta concordância → regex é suficiente. Baixa concordância → IA agrega valor real.

In [ ]:
# ── Normalização base ──────────────────────────────────────────────────────────
def _norm(texto: str) -> str:
    """Remove acentos, lowercase, colapsa espaços."""
    if pd.isna(texto):
        return ""
    t = unicodedata.normalize("NFKD", str(texto).lower()).encode("ascii", "ignore").decode("ascii")
    return re.sub(r"\s+", " ", t)


# ── 1. tipo_documento ──────────────────────────────────────────────────────────
_re_sentenca       = re.compile(r"\bsentenca\b", re.IGNORECASE)
_re_despacho       = re.compile(r"\bdespacho\b", re.IGNORECASE)
_re_decisao_inter  = re.compile(r"\bdecisao\s+interlocutoria\b", re.IGNORECASE)
_re_decisao_inicio = re.compile(r"^decis[ao]\b", re.IGNORECASE)
_re_embargos       = re.compile(r"\bembargos\s+de\s+declaracao\b", re.IGNORECASE)
_re_cumprimento    = re.compile(r"\bcumprimento\s+de\s+sentenca\b", re.IGNORECASE)


def regex_tipo_documento(t: str) -> str:
    n = _norm(t)
    cabecalho = n[:300]

    # Embargos e cumprimento têm prioridade absoluta
    if _re_embargos.search(cabecalho):        return "outro"
    if _re_cumprimento.search(cabecalho):     return "outro"

    # Decisão interlocutória — captura "DECISÃO" sozinho no início
    if _re_decisao_inicio.search(cabecalho[:30]): return "decisão interlocutória"
    if _re_decisao_inter.search(cabecalho):   return "decisão interlocutória"

    if _re_despacho.search(cabecalho):        return "despacho"
    if _re_sentenca.search(cabecalho):        return "sentença"

    # Fallback no texto todo — mesma ordem
    if _re_embargos.search(n):                return "outro"
    if _re_cumprimento.search(n):             return "outro"
    if _re_decisao_inter.search(n):           return "decisão interlocutória"
    if _re_despacho.search(n):                return "despacho"
    if _re_sentenca.search(n):                return "sentença"

    return "não identificado"


# ── 2. fora_do_escopo ──────────────────────────────────────────────────────────
_re_fora = re.compile(
    r"\b(cumprimento\s+de\s+sentenca|embargos\s+de\s+declaracao"
    r"|cancelamento\s+da\s+distribuicao|cancelar\s+a\s+distribuicao"
    r"|art\.\s*924|satisf[ae]ita\s+a\s+obrigacao)\b", re.IGNORECASE
)


def regex_fora_do_escopo(t: str) -> str:
    return "TRUE" if _re_fora.search(_norm(t)) else "FALSE"


# ── 3. justica_gratuita ────────────────────────────────────────────────────────
_re_jg_sim = re.compile(
    r"\b(justica\s+gratuita|gratuidade\s+da\s+justica|gratuidade\s+processual"
    r"|beneficios\s+da\s+(assistencia\s+judiciaria\s+gratuita|justica\s+gratuita)"
    r"|\bajg\b|assistencia\s+judiciaria\s+gratuita"
    r"|defiro\s+.{0,20}gratuidade|concedo\s+.{0,20}gratuidade|gratuidade\s+deferida)", re.IGNORECASE
)
_re_jg_nao = re.compile(
    r"\b(indefiro\s+.{0,30}gratuidade|gratuidade\s+.{0,30}indeferida"
    r"|recolhimento\s+das\s+custas\s+iniciais|sem\s+beneficio\s+da\s+gratuidade)\b",
    re.IGNORECASE
)


def regex_justica_gratuita(t: str) -> str:
    n = _norm(t)
    if _re_jg_nao.search(n): return "não"
    if _re_jg_sim.search(n): return "sim"
    return "não identificado"


# ── 4. rito_processual ─────────────────────────────────────────────────────────
_re_juizado = re.compile(r"\b(juizado\s+especial|lei\s*(n[o°º]?\s*)?9\.099)\b", re.IGNORECASE)
_re_comum   = re.compile(r"\b(procedimento\s+comum(civel)?|rito\s+ordinario)\b", re.IGNORECASE)


def regex_rito_processual(t: str) -> str:
    n = _norm(t)
    if _re_juizado.search(n): return "Juizado Especial"
    if _re_comum.search(n):   return "Procedimento Comum"
    return "não identificado"


# ── 5. tipo_acao ───────────────────────────────────────────────────────────────
_re_superend = re.compile(
    r"\b(superendividamento|lei\s*(n[o°º]?\s*)?14\.181|repactuacao\s+de\s+dividas)\b",
    re.IGNORECASE
)
_re_revisao = re.compile(
    r"\b(revisao\s+(contratual|de\s+juros|de\s+contrato)|juros\s+(remuneratorios|abusivos)"
    r"|capitalizacao|anatocismo|cet\s+abusivo|tabela\s+price|taxa\s+de\s+juros\s+abusiv)\b",
    re.IGNORECASE
)
_re_emprest = re.compile(
    r"\b(emprestimo\s+(nao\s+)?(solicitado|reconhecido|autorizado|contratado)"
    r"|contrato\s+nao\s+reconhecido|descontos?\s+nao\s+(autorizados?|reconhecidos?)"
    r"|nao\s+(contratou|realizou|firmou|celebrou)\s+.{0,20}(emprestimo|contrato)"
    r"|fraude.{0,50}emprestimo|fraude\s+no\s+emprestimo"
    r"|descontos?\s+indevidos?\s+em\s+(beneficio|aposentadoria|salario)"
    r"|contratacao\s+sem\s+autorizacao)\b",
    re.IGNORECASE
)
_re_fraude = re.compile(
    r"\b(vitima\s+de\s+(fraude|golpe|estelionato)|transac[ao]\w*\s+fraudulent\w*"
    r"|operac[ao]\w*\s+fraudulent\w*|fraude\s+bancar\w*|sofreu\s+(fraude|golpe)"
    r"|clonac[ao]\w*\s+de\s+cart[ao]|phishing|motoboy|boa\s+noite\s+cinderela"
    r"|falsa\s+central|golpe\s+do\s+(presente|buque|buque))\b",
    re.IGNORECASE
)
_re_cobranca = re.compile(
    r"\b(cobranca\s+indevida|desconto\s+indevido|tarifa\s+indevida"
    r"|lancamento\s+indevido|cobrado\s+indevidamente|valor\s+cobrado\s+a\s+maior"
    r"|cobranca\s+nao\s+autorizada)\b",
    re.IGNORECASE
)
_re_fora_acao = re.compile(
    r"\b(cumprimento\s+de\s+sentenca|embargos\s+de\s+declaracao"
    r"|cancelamento\s+da\s+distribuicao|litispendencia|coisa\s+julgada)\b",
    re.IGNORECASE
)


def regex_tipo_acao(t: str) -> str:
    n = _norm(t)
    if _re_fora_acao.search(n):  return "fora_do_escopo"
    if _re_superend.search(n):   return "superendividamento"
    if _re_revisao.search(n):    return "revisão contratual"
    if _re_emprest.search(n):    return "empréstimo não reconhecido"
    if _re_fraude.search(n):     return "fraude em conta ou cartão"
    if _re_cobranca.search(n):   return "cobrança indevida"
    return "outro"


# ── 6. contato_previo_banco ────────────────────────────────────────────────────
_re_contato = re.compile(
    r"\b(sac|ouvidoria|procon|reclame\s+aqui"
    r"|reclamac[ao]\s+(administrativa|junto\s+ao?|no\s+banco|na\s+instituicao)"
    r"|protocolo\s+de\s+(atendimento|reclamac[ao])"
    r"|procurou\s+o\s+banco|acionou\s+o\s+banco|comunicou\s+(ao|o)\s+banco"
    r"|tentou\s+(resolver|solucionar)\s+(junto\s+ao?\s+banco|o\s+problema\s+administrativamente)"
    r"|contato\s+administrativo|via\s+administrativa)\b",
    re.IGNORECASE
)


def regex_contato_previo(t: str) -> str:
    return "sim" if _re_contato.search(_norm(t)) else "não"


# ── 7. canal_contato ───────────────────────────────────────────────────────────
_re_reclame   = re.compile(r"\breclame\s+aqui\b", re.IGNORECASE)
_re_procon    = re.compile(r"\b(procon|decon)\b", re.IGNORECASE)
_re_ouvidoria = re.compile(r"\bouvidoria\b", re.IGNORECASE)
_re_sac       = re.compile(r"\b(sac|servico\s+de\s+atendimento\s+ao\s+consumidor)\b", re.IGNORECASE)
_re_agencia   = re.compile(
    r"(compareceu|dirigiu.se|foi\s+(ate|a\s+uma?)|presencialmente\s+na"
    r"|atendimento\s+presencial)\s+(a\s+)?agencia", re.IGNORECASE
)


def regex_canal_contato(t: str) -> str:
    n = _norm(t)
    if _re_reclame.search(n):   return "Reclame Aqui"
    if _re_procon.search(n):    return "Procon"
    if _re_ouvidoria.search(n): return "Ouvidoria"
    if _re_sac.search(n):       return "SAC"
    if _re_agencia.search(n):   return "Agência"
    return "não identificado"


# ── 8. mencao_reclame_aqui ─────────────────────────────────────────────────────
def regex_mencao_reclame(t: str) -> str:
    return "sim" if _re_reclame.search(_norm(t)) else "não"


# ── 9. boletim_de_ocorrencia ───────────────────────────────────────────────────
_re_bo = re.compile(
    r"\b(boletim\s+de\s+ocorrencia|registro\s+(de\s+)?(ocorrencia|policial)"
    r"|B\.O\.\s*n[o°º]?|b\.o\.\s+n[o°º]?)\b",
    re.IGNORECASE
)


def regex_boletim(t: str) -> str:
    return "sim" if _re_bo.search(_norm(t)) else "não"


# ── 10. resultado_julgamento ───────────────────────────────────────────────────
def regex_resultado(t: str) -> str:
    n = _norm(t)

    # Busca trecho após JULGO
    m_julgo = re.search(r"\b(?:julgo)\b(.{0,300})", n)
    trecho  = m_julgo.group(1) if m_julgo else n[:300]

    # Busca separada para HOMOLOGO (acordo)
    m_acordo = re.search(r"\bhomologo\b(.{0,300})", n)
    trecho_acordo = m_acordo.group(1) if m_acordo else ""

    if re.search(r"parcialmente\s+procedente", trecho):
        return "parcialmente procedente"
    if re.search(r"improcedent(?:e|es)", trecho):
        return "improcedente"
    if re.search(r"procedent(?:e|es)", trecho):
        return "procedente"
    if re.search(r"prescric[ao]|decadenci", trecho):
        return "extinto com mérito prescrição"
    if re.search(r"acordo|transac[ao]", trecho_acordo):
        return "extinto acordo"
    if re.search(r"extint[ao]", trecho):
        return "extinto sem mérito"
    if re.search(r"indefiro\s+a\s+peticao\s+inicial|cancelamento\s+da\s+distribuicao", trecho):
        return "extinto sem mérito"

    return "não identificado"


# ── 11. culpa_atribuida ────────────────────────────────────────────────────────
_re_compart  = re.compile(
    r"\b(culpa\s+(concorrente|compartilhada|reciproca)|concorrencia\s+de\s+culpas?)\b",
    re.IGNORECASE
)
_re_terceiro = re.compile(
    r"\b(culpa\s+(exclusiva\s+)?de\s+terceiro|fraude\s+(praticada\s+)?por\s+terceiro"
    r"|estelionato\s+(praticado\s+)?por\s+terceiro)\b",
    re.IGNORECASE
)
_re_banco = re.compile(
    r"\b(falha\s+(na\s+)?prestacao\s+de\s+servicos?"
    r"|responsabilidade\s+.{0,20}(banco|requerido|reu)\s+.{0,20}reconhecida"
    r"|condeno\s+.{0,10}(banco|requerido|reu)"
    r"|dano\s+causado\s+pelo\s+(banco|requerido|reu))\b",
    re.IGNORECASE
)
_re_consumidor = re.compile(
    r"\b(culpa\s+(exclusiva\s+)?d[oa]\s+(autor[a]?|requerente|consumidor[a]?)"
    r"|ausencia\s+de\s+(falha|vicio|defeito|conduta\s+ilicita)"
    r"|nao\s+h[ao]\s+(falha|vicio|defeito)\s+na\s+prestacao)\b",
    re.IGNORECASE
)


def regex_culpa(t: str) -> str:
    n = _norm(t)
    if _re_compart.search(n):    return "compartilhada"
    if _re_terceiro.search(n):   return "terceiro"
    if _re_banco.search(n):      return "banco"
    if _re_consumidor.search(n): return "consumidor"
    # Heurística: improcedente → consumidor; procedente → banco
    if re.search(r"julgo\s+improcedente", n): return "consumidor"
    if re.search(r"julgo\s+procedente|\bcondeno\b", n): return "banco"
    return "não identificado"


# ── 12 & 13. valores monetários ────────────────────────────────────────────────
def _extrair_valor(texto: str, campo_re: str) -> float:
    """Extrai R$ X no trecho de ±150 chars ao redor do campo indicado."""
    t = str(texto)
    for pos in [m.start() for m in re.finditer(campo_re, t, re.IGNORECASE)]:
        trecho = t[max(0, pos - 150): pos + 200]
        for v in re.findall(r"R\$\s*([\d.,]+)", trecho):
            try:
                return float(v.replace(".", "").replace(",", "."))
            except ValueError:
                pass
    return 0.0


def regex_valor_morais(t: str) -> float:
    return _extrair_valor(t, r"danos\s+morais")


def regex_valor_materiais(t: str) -> float:
    return _extrair_valor(t, r"danos\s+materiais")


# ── 14. repetição_indébito ─────────────────────────────────────────────────────
_re_dobro   = re.compile(
    r"\b(repeticao\s+em\s+dobro|devolucao\s+em\s+dobro|indebito\s+em\s+dobro|restituicao\s+em\s+dobro)\b",
    re.IGNORECASE
)
_re_simples = re.compile(
    r"\b(repeticao\s+(do|de)\s+indebito|restituicao\s+simples|devolucao\s+(do\s+)?valor"
    r"|repeticao\s+simples|restituicao\s+do\s+valor)\b",
    re.IGNORECASE
)


def regex_repeticao_indebito(t: str) -> str:
    n = _norm(t)
    if _re_dobro.search(n):   return "dobro"
    if _re_simples.search(n): return "simples"
    return "não identificado"


# ── Aplica tudo ────────────────────────────────────────────────────────────────
decisao = df_curto["decisao"].fillna("")

df_curto["tipo_documento_regex"]        = decisao.apply(regex_tipo_documento)
df_curto["fora_do_escopo_regex"]        = decisao.apply(regex_fora_do_escopo)
df_curto["justica_gratuita_regex"]      = decisao.apply(regex_justica_gratuita)
df_curto["rito_processual_regex"]       = decisao.apply(regex_rito_processual)
df_curto["tipo_acao_regex"]             = decisao.apply(regex_tipo_acao)
df_curto["contato_previo_banco_regex"]  = decisao.apply(regex_contato_previo)
df_curto["canal_contato_regex"]         = decisao.apply(regex_canal_contato)
df_curto["mencao_reclame_aqui_regex"]   = decisao.apply(regex_mencao_reclame)
df_curto["boletim_de_ocorrencia_regex"] = decisao.apply(regex_boletim)
df_curto["resultado_julgamento_regex"]  = decisao.apply(regex_resultado)
df_curto["culpa_atribuida_regex"]       = decisao.apply(regex_culpa)
df_curto["valor_danos_morais_regex"]    = decisao.apply(regex_valor_morais)
df_curto["valor_danos_materiais_regex"] = decisao.apply(regex_valor_materiais)
df_curto["repeticao_indebito_regex"]    = decisao.apply(regex_repeticao_indebito)

print(f"Colunas _regex adicionadas: {len([c for c in df_curto.columns if c.endswith('_regex')])}")

In [ ]:
# resultados_sem_cond = ["improcedente", "extinto sem mérito", 
#                         "extinto com mérito prescrição", "extinto acordo"]

# mask_sem_cond = df_curto["resultado_julgamento_ia"].isin(resultados_sem_cond)

# df_curto.loc[
#     mask_sem_cond & (df_curto["repetição_indébito_regex"] == "não identificado"),
#     "repetição_indébito_regex"
# ] = "não aplicável"

Comparacao:

In [ ]:
## Comparação IA vs Regex

# Campos comparáveis (exclui valores numéricos — têm métrica própria)
campos_categoricos = [
    "tipo_documento",
    "justica_gratuita",
    "rito_processual",
    "tipo_acao",
    "contato_previo_banco",
    "canal_contato",
    "mencao_reclame_aqui",
    "boletim_de_ocorrencia",
    "resultado_julgamento",
    "culpa_atribuida",
    "repetição_indébito",
]

# Exclui linhas com ERRO na IA e fora do escopo
mask_validos = (
    ~df_curto["resultado_julgamento_ia"].astype(str).str.startswith("ERRO") &
    (df_curto["fora_do_escopo_ia"] == False)
)
sub = df_curto[mask_validos].copy()

print(f"Linhas analisadas (sem erro, sem fora do escopo): {len(sub)}\n")
print(f"{'CAMPO':<30} {'IGUAIS':>7} {'DIFER.':>7} {'ACORDO %':>9}")
print("-" * 58)

rows = []
for campo in campos_categoricos:
    col_ia    = f"{campo}_ia"
    col_regex = f"{campo}_regex"
    if col_ia not in sub.columns or col_regex not in sub.columns:
        continue

    ia    = sub[col_ia].astype(str).str.strip().str.lower()
    regex = sub[col_regex].astype(str).str.strip().str.lower()

    iguais = (ia == regex).sum()
    total  = len(sub)
    acordo = iguais / total * 100

    print(f"{campo:<30} {iguais:>7} {total - iguais:>7} {acordo:>8.1f}%")
    rows.append({"campo": campo, "iguais": iguais, "diferentes": total - iguais, "acordo_%": round(acordo, 1)})

df_concordancia = pd.DataFrame(rows).sort_values("acordo_%", ascending=False)

# Comparação de valores numéricos
print("\n--- Valores numéricos (só linhas com condenação > 0 em ambos) ---")
for campo_val in ["valor_danos_morais", "valor_danos_materiais"]:
    col_ia    = f"{campo_val}_ia"
    col_regex = f"{campo_val}_regex"
    ambos_pos = sub[(sub[col_ia] > 0) & (sub[col_regex] > 0)]
    if len(ambos_pos) == 0:
        print(f"{campo_val}: sem casos com ambos > 0")
        continue
    diff_pct = ((ambos_pos[col_ia] - ambos_pos[col_regex]).abs() / ambos_pos[col_ia] * 100)
    print(f"{campo_val}: n={len(ambos_pos)}, diferença média={diff_pct.mean():.1f}%, mediana={diff_pct.median():.1f}%")

# Inspeção manual das divergências
print("\n--- Inspeção: campos com menor acordo ---")
campo_pior = df_concordancia.iloc[-1]["campo"]
col_ia    = f"{campo_pior}_ia"
col_regex = f"{campo_pior}_regex"
divergentes = sub[sub[col_ia].astype(str).str.lower() != sub[col_regex].astype(str).str.lower()]
print(f"\nCampo '{campo_pior}' — {len(divergentes)} divergências. Amostra de 5:\n")
display(
    divergentes[["id_processo", col_ia, col_regex]]
    .head(5)
    .rename(columns={col_ia: "IA", col_regex: "Regex"})
)

In [ ]:
# Exportar xlsx com colunas _ia e _regex intercaladas

colunas_base = [c for c in df_curto.columns if not c.endswith("_ia") and not c.endswith("_regex")]

campos_ia = [c for c in df_curto.columns if c.endswith("_ia")]
campos_regex = [c.replace("_ia", "_regex") for c in campos_ia if c.replace("_ia", "_regex") in df_curto.columns]

colunas_intercaladas = []
for col_ia in campos_ia:
    colunas_intercaladas.append(col_ia)
    col_regex = col_ia.replace("_ia", "_regex")
    if col_regex in df_curto.columns:
        colunas_intercaladas.append(col_regex)
        


ordem_final = colunas_base + colunas_intercaladas


pasta_saida = "output"
os.makedirs(pasta_saida, exist_ok=True)
arquivo_saida = os.path.join(pasta_saida, "curto_ia_regex.xlsx")
df_curto[ordem_final].to_excel(arquivo_saida, index=False, sheet_name="Processos")



print(f"Arquivo salvo: output/curto_ia_regex.xlsx")
print(f"Total colunas: {len(ordem_final)} ({len(colunas_base)} base + {len(colunas_intercaladas)} IA/regex)")
print("\nOrdem das colunas IA/regex:")
for i in range(0, len(colunas_intercaladas), 2):
    ia = colunas_intercaladas[i]
    rx = colunas_intercaladas[i+1] if i+1 < len(colunas_intercaladas) else "—"
    print(f"  {ia:<35} | {rx}")

In [ ]:
# Carrega o arquivo com IA e Regex para análise
df_analise = pd.read_excel("output/curto_ia_regex.xlsx", sheet_name="Processos")

In [ ]:
df_ana_t_doc = df_analise.copy()

# Divergências em tipo_documento
div = df_ana_t_doc[df_ana_t_doc["tipo_documento_ia"] != df_ana_t_doc["tipo_documento_regex"]].reset_index(drop=True)

print(f"Total divergências tipo_documento: {len(div)}")
display(div[["decisao","tipo_documento_ia", "tipo_documento_regex"]])

df_ana_t_doc=div[["decisao","tipo_documento_ia", "tipo_documento_regex"]]

arquivo_saida = os.path.join(pasta_saida, "df_ana_t_doc.xlsx")
df_ana_t_doc.to_excel(arquivo_saida, index=False, sheet_name="Processos")


In [ ]:
df_tip_a = df_analise.copy()

# Divergências em tipo_documento
div = df_tip_a[df_tip_a["tipo_acao_ia"] != df_tip_a["tipo_acao_regex"]].reset_index(drop=True)

print(f"Total divergências tipo_documento: {len(div)}")
display(div[["decisao","tipo_acao_ia", "tipo_acao_regex"]])

df_tip_a=div[["decisao","tipo_acao_ia", "tipo_acao_regex"]]

arquivo_saida = os.path.join(pasta_saida, "df_tip_a.xlsx")
df_tip_a.to_excel(arquivo_saida, index=False, sheet_name="Processos")
